In [1]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 46.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
MODEL_NAMES = [
    "bert-base-uncased",
    "roberta-base",
    "distilbert-base-uncased"
]

In [6]:
df = pd.read_csv("/content/drive/MyDrive/Dataset.csv")

In [7]:
df = df[['Recipe', 'Approximate Calorie per Serving']].dropna()

In [8]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(df['Recipe'].tolist(), df['Approximate Calorie per Serving'].values, test_size=0.3, random_state=42)
val_texts, test_texts, val_labels, test_labels = train_test_split(temp_texts, temp_labels, test_size=0.5, random_state=42)

In [9]:
class RecipeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

In [10]:
class TransformerRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden_size = self.transformer.config.hidden_size
        self.regressor = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        if hasattr(outputs, 'pooler_output'):
            cls_output = outputs.pooler_output
        else:
            cls_output = outputs.last_hidden_state[:, 0]
        preds = self.regressor(cls_output).squeeze(-1)
        loss = None
        if labels is not None:
            loss = nn.MSELoss()(preds, labels)
        return {"loss": loss, "logits": preds}

In [11]:
val_preds_list, test_preds_list = [], []

In [12]:
from transformers import TrainingArguments

In [13]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [14]:
for model_name in MODEL_NAMES:
    print(f"\n\n Training {model_name}\n\n")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
    val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)
    test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

    train_dataset = RecipeDataset(train_encodings, train_labels)
    val_dataset = RecipeDataset(val_encodings, val_labels)
    test_dataset = RecipeDataset(test_encodings, test_labels)

    model = TransformerRegressor(model_name).to(device)

    training_args = TrainingArguments(
      output_dir="./results",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=3,
      weight_decay=0.01,
      logging_dir="./logs",
      logging_steps=500
    )

    trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset
    )

    trainer.train()

    trainer.evaluate()

    val_preds = trainer.predict(val_dataset).predictions
    test_preds = trainer.predict(test_dataset).predictions

    val_preds_list.append(val_preds)
    test_preds_list.append(test_preds)



 Training bert-base-uncased




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss


In [15]:
X_val_stack = np.stack(val_preds_list, axis=1)
X_test_stack = np.stack(test_preds_list, axis=1)

In [16]:
meta_model = Ridge()
meta_model.fit(X_val_stack, val_labels)
stacked_preds = meta_model.predict(X_test_stack)

In [17]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [18]:
def evaluate_regression(true, predicted):
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(true, predicted)
    r2 = r2_score(true, predicted)
    mape = np.mean(np.abs((true - predicted) / (true + 1e-8))) * 100

    print("\nEvaluation Metrics:")
    print(f"MSE  (Mean Squared Error):      {mse:.2f}")
    print(f"RMSE (Root MSE):                {rmse:.2f}")
    print(f"MAE  (Mean Absolute Error):     {mae:.2f}")
    print(f"R²   (R-squared):                {r2:.4f}")
    print(f"MAPE (Mean Absolute % Error):   {mape:.2f}%")

In [19]:
evaluate_regression(test_labels, stacked_preds)


Evaluation Metrics:
MSE  (Mean Squared Error):      12345.55
RMSE (Root MSE):                111.11
MAE  (Mean Absolute Error):     89.16
R²   (R-squared):                -0.0000
MAPE (Mean Absolute % Error):   34.80%
